# Hype Open Edge

**Hipótesis**: Las señales `confirmed / accelerating / spike` de `hype_metrics` en los primeros 60 minutos del mercado (9:30-10:30 ET) predicen movimientos direccionales medibles.  
Si compramos al precio de señal con un trailing stop, ¿hay edge positivo?

## Pipeline correcto
- **Fuente de señal**: `hype_metrics` (confirmed / accelerating / spike)
- **Precio de entrada**: `close_price` del row de señal (precio de mercado en ese minuto)
- **Precios futuros**: `market_bars` — barras 1-min del prod DB
- **Independencia**: un evento por (ticker, date) = primera señal del día
- **Slippage**: 0.5% deducido de todos los retornos

## Advertencia de muestra
Los datos ricos (Apr 14-17, 2026) son solo 4 días de trading.  
Cualquier resultado debe interpretarse como **exploratorio** con n pequeño.
No se pueden extraer conclusiones estadísticamente válidas con 4 días.

In [ ]:
import sys, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from helpers import load_hype_metrics, load_market_bars
from helpers.forward_returns import BarsIndex, edge_summary

pd.set_option('display.float_format', '{:.4f}'.format)

# ── Config ─────────────────────────────────────────────────
SIGNALS      = ('confirmed', 'accelerating', 'spike')
HORIZONS     = [15, 30, 60]          # minutos
SLIPPAGE_PCT = 0.005                 # 0.5%
OPEN_WINDOW  = (0, 60)               # minutos desde apertura (9:30-10:30)

print('Configuración cargada')

## 1. Carga de datos

In [ ]:
hype = load_hype_metrics(signals=SIGNALS)
bars = load_market_bars()

print(f'hype_metrics: {len(hype):,} filas | {hype["ticker"].nunique()} tickers | {hype["date"].nunique()} días')
print(f'  Rango: {hype["date"].min()} → {hype["date"].max()}')
print(f'  Señales: {dict(hype["signal"].value_counts())}')
print()
print(f'market_bars: {len(bars):,} filas | {bars["ticker"].nunique()} tickers | {bars["date"].nunique()} días')
print(f'  Rango: {bars["date"].min()} → {bars["date"].max()}')

## 2. Filtro: ventana de apertura + independencia

In [ ]:
# Filtrar a ventana de apertura
hype_open = hype[
    (hype['minutes_since_open'] >= OPEN_WINDOW[0]) &
    (hype['minutes_since_open'] <  OPEN_WINDOW[1])
].copy()

print(f'Señales en ventana {OPEN_WINDOW[0]}-{OPEN_WINDOW[1]}min: {len(hype_open):,}')
print(f'Tickers únicos: {hype_open["ticker"].nunique()}')
print(f'Días: {sorted(hype_open["date"].unique())}')
print()

# Resumen por día
print('Señales por día en ventana de apertura:')
display(hype_open.groupby('date').agg(
    n_signals=('ticker', 'count'),
    n_tickers=('ticker', 'nunique'),
    signals=('signal', lambda x: dict(x.value_counts()))
))

In [ ]:
# Primera señal por (ticker, date) — independencia
hype_indep = (
    hype_open
    .sort_values('ts')
    .groupby(['ticker', 'date'], as_index=False)
    .first()
)

print(f'Eventos independientes (primera señal por ticker-día): {len(hype_indep)}')
print(f'Tickers cubiertos por market_bars:')

bars_keys = set(zip(bars['ticker'], bars['date']))
hype_indep['has_bars'] = hype_indep.apply(
    lambda r: (r['ticker'], r['date']) in bars_keys, axis=1
)
print(f'  Con barras: {hype_indep["has_bars"].sum()} / {len(hype_indep)}')

display(hype_indep.groupby('date')[['has_bars']].agg(['sum', 'count']))

## 3. Forward returns con market_bars

In [ ]:
# Construir índice de market_bars
bars_idx = BarsIndex(bars.rename(columns={'bar_ts': 'bar_ts'}))

# Calcular forward returns
# Entrada: close_price del row de señal (precio de mercado al momento de la señal)
# Salida: close del bar más cercano a signal_ts + horizon

records = []
for _, row in hype_indep.iterrows():
    if not row['has_bars']:
        continue
    
    day_bars = bars_idx.get(row['ticker'], row['date'])
    if day_bars is None:
        continue
    
    signal_ts    = row['ts']
    entry_price  = row['close_price']   # precio al momento de la señal
    
    if entry_price is None or entry_price <= 0:
        continue
    
    rec = {
        'ticker'     : row['ticker'],
        'date'       : row['date'],
        'signal'     : row['signal'],
        'signal_ts'  : signal_ts,
        'entry_price': entry_price,
        'rel_volume' : row['rel_volume'],
        'delta_5m'   : row['delta_5m'],
        'hype_cum'   : row['hype_cum'],
        'mso'        : row['minutes_since_open'],
    }
    
    for h in HORIZONS:
        exit_target = signal_ts + pd.Timedelta(minutes=h)
        exit_cands  = day_bars[day_bars['bar_ts'] >= exit_target]
        exit_bar    = exit_cands.iloc[0] if len(exit_cands) > 0 else day_bars.iloc[-1]
        exit_price  = exit_bar['close']
        
        ret = (exit_price - entry_price) / entry_price
        
        # MFE/MAE
        window = day_bars[
            (day_bars['bar_ts'] >= signal_ts) &
            (day_bars['bar_ts'] <= exit_bar['bar_ts'])
        ]
        mfe = (window['high'].max()  - entry_price) / entry_price if len(window) > 0 else None
        mae = (window['low'].min()   - entry_price) / entry_price if len(window) > 0 else None
        
        rec[f'ret_{h}m']  = ret
        rec[f'mfe_{h}m']  = mfe
        rec[f'mae_{h}m']  = mae
    
    records.append(rec)

df = pd.DataFrame(records)
print(f'Eventos con forward returns calculados: {len(df)}')

# Aplicar slippage
for h in HORIZONS:
    df[f'ret_net_{h}m'] = df[f'ret_{h}m'] - SLIPPAGE_PCT

print(df[['ticker', 'date', 'signal', 'entry_price', 'ret_30m', 'ret_net_30m']].head(10))

## 4. Edge global

In [ ]:
print('=== Edge BRUTO (sin slippage) ===')
display(edge_summary(df, HORIZONS, col_prefix='ret'))

print()
print('=== Edge NETO (con slippage 0.5%) ===')
display(edge_summary(df, HORIZONS, col_prefix='ret_net'))

## 5. Edge por tipo de señal

In [ ]:
for sig in sorted(df['signal'].unique()):
    sub = df[df['signal'] == sig]
    if len(sub) < 3:
        continue
    print(f'\n=== {sig.upper()} (n={len(sub)}) ===')
    display(edge_summary(sub, HORIZONS, col_prefix='ret_net'))

## 6. Distribución de retornos

In [ ]:
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5 * len(HORIZONS), 4))
if len(HORIZONS) == 1:
    axes = [axes]

for ax, h in zip(axes, HORIZONS):
    col = f'ret_net_{h}m'
    s   = df[col].dropna() * 100
    wr  = (s > 0).mean()
    ax.hist(s, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(0, color='red', lw=2)
    ax.axvline(s.mean(), color='orange', lw=1.5, linestyle='--', label=f'mean={s.mean():.1f}%')
    ax.set_title(f'T+{h}m  WR={wr:.0%}  n={len(s)}')
    ax.set_xlabel('Retorno neto (%)')
    ax.legend(fontsize=9)

plt.suptitle('Distribución de retornos netos — señales apertura', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Curvas de precio intradiarias (primeros 60 min)

In [ ]:
# Normalizar cada ticker a 1.0 en el momento de la señal y mostrar las curvas
# Solo tickers de Apr 14 para tener muestra razonable

date_focus = df['date'].value_counts().idxmax()
df_day = df[df['date'] == date_focus].copy()

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(df_day)))

# Ordenar por retorno T+60m para dar color (rojo=perdedor, verde=ganador)
df_day = df_day.sort_values('ret_net_60m', na_position='last') if 'ret_net_60m' in df_day.columns else df_day

for (_, row), color in zip(df_day.iterrows(), colors):
    day_bars = bars_idx.get(row['ticker'], row['date'])
    if day_bars is None:
        continue
    
    signal_ts = row['signal_ts']
    window = day_bars[
        (day_bars['bar_ts'] >= signal_ts) &
        (day_bars['bar_ts'] <= signal_ts + pd.Timedelta(minutes=60))
    ]
    if len(window) < 3:
        continue
    
    norm = window['close'] / row['entry_price']  # normalizado a 1.0
    mins = (window['bar_ts'] - signal_ts).dt.total_seconds() / 60
    ax.plot(mins, norm, alpha=0.5, color=color, lw=1)

ax.axhline(1.0, color='black', lw=1.5, linestyle='--')
ax.axhline(1.0 - SLIPPAGE_PCT, color='gray', lw=1, linestyle=':', label='entrada efectiva (slippage)')
ax.set_xlabel('Minutos desde señal')
ax.set_ylabel('Precio normalizado (1.0 = precio señal)')
ax.set_title(f'Curvas de precio: {len(df_day)} señales en {date_focus}\n(rojo=perdedor, verde=ganador al T+60m)')
ax.legend()
plt.tight_layout()
plt.show()

## 8. MFE vs MAE — sizing de stops y targets

In [ ]:
h = 30  # horizonte de referencia
sub = df[df[f'mfe_{h}m'].notna() & df[f'mae_{h}m'].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(sub[f'mae_{h}m'] * 100, sub[f'mfe_{h}m'] * 100, alpha=0.6, s=40)
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel(f'MAE_{h}m (%)  ← drawdown máximo')
ax.set_ylabel(f'MFE_{h}m (%)  ↑ ganancia máxima')
ax.set_title('MFE vs MAE')

ax = axes[1]
mfe_p = sub[f'mfe_{h}m'] * 100
mae_p = sub[f'mae_{h}m'] * 100
percentiles = [10, 25, 50, 75, 90]
ax.barh(
    [f'MFE p{p}' for p in percentiles],
    [np.percentile(mfe_p.dropna(), p) for p in percentiles],
    color='green', alpha=0.6, label='MFE'
)
ax.barh(
    [f'MAE p{p}' for p in percentiles],
    [np.percentile(mae_p.dropna(), p) for p in percentiles],
    color='red', alpha=0.6, label='MAE'
)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('%')
ax.set_title(f'Percentiles MFE/MAE T+{h}m')
ax.legend()

plt.tight_layout()
plt.show()

print(f'\nT+{h}m MFE percentiles: p25={np.percentile(mfe_p.dropna(),25):.1f}% | p50={np.percentile(mfe_p.dropna(),50):.1f}% | p75={np.percentile(mfe_p.dropna(),75):.1f}%')
print(f'T+{h}m MAE percentiles: p25={np.percentile(mae_p.dropna(),25):.1f}% | p50={np.percentile(mae_p.dropna(),50):.1f}% | p75={np.percentile(mae_p.dropna(),75):.1f}%')

## 9. Trailing stop simulado

Simulamos una salida por trailing stop: si el precio cae X% desde el máximo alcanzado, salimos.

In [ ]:
def simulate_trailing_stop(row, bars_idx, trail_pct=0.03, max_minutes=60):
    """Simula trailing stop. Retorna (ret, exit_min) o (None, None)."""
    day_bars = bars_idx.get(row['ticker'], row['date'])
    if day_bars is None:
        return None, None
    
    signal_ts   = row['signal_ts']
    entry_price = row['entry_price']
    end_ts      = signal_ts + pd.Timedelta(minutes=max_minutes)
    
    window = day_bars[
        (day_bars['bar_ts'] > signal_ts) &
        (day_bars['bar_ts'] <= end_ts)
    ].reset_index(drop=True)
    
    if len(window) == 0:
        return None, None
    
    high_water = entry_price
    for _, bar in window.iterrows():
        high_water = max(high_water, bar['high'])
        trail_stop = high_water * (1 - trail_pct)
        if bar['low'] <= trail_stop:
            exit_price = trail_stop
            exit_min   = (bar['bar_ts'] - signal_ts).total_seconds() / 60
            return (exit_price - entry_price) / entry_price, exit_min
    
    # No tocó el trailing stop → salimos al cierre del último bar
    exit_price = window.iloc[-1]['close']
    exit_min   = (window.iloc[-1]['bar_ts'] - signal_ts).total_seconds() / 60
    return (exit_price - entry_price) / entry_price, exit_min


# Sweep de trailing stops
trail_pcts = [0.02, 0.03, 0.05, 0.07]
summary_rows = []

for trail in trail_pcts:
    rets = []
    for _, row in df.iterrows():
        ret, _ = simulate_trailing_stop(row, bars_idx, trail_pct=trail, max_minutes=60)
        if ret is not None:
            rets.append(ret - SLIPPAGE_PCT)
    
    if not rets:
        continue
    
    s    = pd.Series(rets)
    wins = s[s > 0]
    loss = s[s < 0]
    pf   = wins.sum() / abs(loss.sum()) if len(loss) > 0 else np.inf
    summary_rows.append({
        'trailing_stop': f'{trail*100:.0f}%',
        'n'            : len(s),
        'wr'           : f'{(s > 0).mean():.0%}',
        'profit_factor': round(pf, 2),
        'avg_ret_pct'  : f'{s.mean()*100:.2f}%',
        'median_ret_pct': f'{s.median()*100:.2f}%',
    })

print('=== Trailing Stop Simulado (señales apertura, T+60m máx, neto de slippage 0.5%) ===')
display(pd.DataFrame(summary_rows))

## 10. Inventario de sesgos

| Sesgo | Estado | Descripción |
|---|---|---|
| Independencia | ✅ Controlado | Una señal por ticker-día |
| Survivorship bias | ⚠️ Parcial | market_bars solo tiene ~40 tickers/día que el dashboard rastreó activamente |
| Régimen único | ⚠️ Crítico | Datos Apr 14-17 = semana de crash (tariff week). No representativo de mercado normal |
| Lookahead | ✅ Controlado | Entrada al precio de señal, no al precio futuro |
| Slippage | ✅ Modelado | 0.5% deducido |
| Tamaño de muestra | ❌ Insuficiente | n≈39 (Apr 14) + n<10 otros días = evidencia solo exploratoria |

**Conclusión**: este análisis es exploratorio, no validatorio. Se necesitan más días con datos ricos en market_bars para poder concluir algo estadísticamente válido.

In [ ]:
print('=== RESUMEN FINAL ===')
print(f'  Días con datos: {sorted(df["date"].unique())}')
print(f'  N total (indep): {len(df)}')
print(f'  N por día: {dict(df.groupby("date").size())}')
print()

for h in HORIZONS:
    col = f'ret_net_{h}m'
    s   = df[col].dropna() * 100
    wr  = (s > 0).mean()
    pf_val = s[s>0].sum() / abs(s[s<0].sum()) if len(s[s<0]) > 0 else np.inf
    print(f'  T+{h}m → WR={wr:.0%}  PF={pf_val:.2f}  avg={s.mean():.2f}%  n={len(s)}')

print()
print('ADVERTENCIA: Solo 4 días de datos, régimen de crash (Apr 14-17 tariff week).')
print('Los resultados NO son estadísticamente válidos ni generalizables.')